In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Write a program to invert the colors of an image. The image is
  represented as a 1D array of RGBA (Red, Green, Blue, Alpha) values, where each
  component is an 8-bit unsigned integer (<code>unsigned char</code>).
</p>

<p>
  Color inversion is performed by subtracting each color component (R, G, B)
  from 255. The Alpha component should remain unchanged.
</p>

<p>
  The input array
  <code>image</code> will contain <code>width * height * 4</code> elements. The
  first 4 elements represent the RGBA values of the top-left pixel, the next 4
  elements represent the pixel to its right, and so on.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>
    The final result must be stored in the array
    <code>image</code>
  </li>
</ul>

<h2>Example 1:</h2>
<pre>
Input: image = [255, 0, 128, 255, 0, 255, 0, 255], width=1, height=2
Output: [0, 255, 127, 255, 255, 0, 255, 255]
</pre>

<h2>Example 2:</h2>
<pre>
Input: image = [10, 20, 30, 255, 100, 150, 200, 255], width=2, height=1
Output: [245, 235, 225, 255, 155, 105, 55, 255]
</pre>

<h2>Constraints</h2>

<ul>
  <li>1 &le; <code>width</code> &le; 4096</li>
  <li>1 &le; <code>height</code> &le; 4096</li>
  <li><code>width * height</code> &le; 8,388,608.</li>

  <li>Performance is measured with <code>height</code> = 5,120, <code>width</code> = 4,096</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__global__ void invert_kernel(unsigned char* image, int width, int height) {}
// image_input, image_output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(unsigned char* image, int width, int height) {
    int threadsPerBlock = 256;
    int blocksPerGrid = (width * height + threadsPerBlock - 1) / threadsPerBlock;

    invert_kernel<<<blocksPerGrid, threadsPerBlock>>>(image, width, height);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# image are tensors on the GPU
@cute.jit
def solve(image: cute.Tensor, width: cute.Int32, height: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# image is a tensor on the GPU
@jax.jit
def solve(image: jax.Array, width: int, height: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def invert_kernel(image: UnsafePointer[UInt8, MutExternalOrigin], width: Int32, height: Int32):
    pass


# image is a device pointer (i.e. pointer to memory on the GPU)
@export
def solve(image: UnsafePointer[UInt8, MutExternalOrigin], width: Int32, height: Int32) raises:
    var threadsPerBlock: Int32 = 256
    var ctx = DeviceContext()

    var total_pixels = width * height
    var blocksPerGrid = ceildiv(total_pixels, threadsPerBlock)

    var _kernel = ctx.compile_function[invert_kernel, invert_kernel]()
    ctx.enqueue_function(
        _kernel, image, width, height, grid_dim=blocksPerGrid, block_dim=threadsPerBlock
    )

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution.py
import torch


# image is a tensor on the GPU
def solve(image: torch.Tensor, width: int, height: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


@triton.jit
def invert_kernel(image, width, height, BLOCK_SIZE: tl.constexpr):
    pass


# image is a tensor on the GPU
def solve(image: torch.Tensor, width: int, height: int):
    BLOCK_SIZE = 1024
    n_pixels = width * height
    grid = (triton.cdiv(n_pixels, BLOCK_SIZE),)

    invert_kernel[grid](image, width, height, BLOCK_SIZE)


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Color Inversion", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(self, image: torch.Tensor, width: int, height: int):
        assert image.shape == (height * width * 4,)
        assert image.dtype == torch.uint8

        # Reshape to (height, width, 4) for easier processing
        image_reshaped = image.view(height, width, 4)

        # Invert RGB channels (first 3 channels), keep alpha unchanged
        image_reshaped[:, :, :3] = 255 - image_reshaped[:, :, :3]

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "image": (ctypes.POINTER(ctypes.c_ubyte), "inout"),
            "width": (ctypes.c_int, "in"),
            "height": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        width, height = 1, 2
        image = torch.tensor([255, 0, 128, 255, 0, 255, 0, 255], device="cuda", dtype=torch.uint8)
        return {
            "image": image,
            "width": width,
            "height": height,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        return [
            {
                "image": torch.tensor(
                    [
                        [[255, 0, 0, 255], [0, 255, 0, 255]],
                        [[0, 0, 255, 255], [128, 128, 128, 255]],
                    ],
                    dtype=torch.uint8,
                    device="cuda",
                ).flatten(),
                "width": 2,
                "height": 2,
            },
            {
                "image": torch.tensor(
                    [[[100, 50, 200, 255]]], dtype=torch.uint8, device="cuda"
                ).flatten(),
                "width": 1,
                "height": 1,
            },
            {
                "image": torch.zeros((3, 4, 4), dtype=torch.uint8, device="cuda").flatten(),
                "width": 4,
                "height": 3,
            },
            {
                "image": torch.full((5, 3, 4), 255, dtype=torch.uint8, device="cuda").flatten(),
                "width": 3,
                "height": 5,
            },
            {
                "image": torch.tensor(
                    [
                        [[10, 20, 30, 50], [40, 50, 60, 100]],
                        [[70, 80, 90, 150], [100, 110, 120, 200]],
                    ],
                    dtype=torch.uint8,
                    device="cuda",
                ).flatten(),
                "width": 2,
                "height": 2,
            },
            {
                "image": torch.randint(0, 256, (64 * 64 * 4,), dtype=torch.uint8, device="cuda"),
                "width": 64,
                "height": 64,
            },
            {
                "image": torch.randint(0, 256, (32 * 64 * 4,), dtype=torch.uint8, device="cuda"),
                "width": 64,
                "height": 32,
            },
        ]

    def generate_performance_test(self) -> Dict[str, Any]:
        width, height = 4096, 5120
        size = width * height * 4
        return {
            "image": torch.randint(0, 256, (size,), device="cuda", dtype=torch.uint8),
            "width": width,
            "height": height,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
